<a href="https://colab.research.google.com/github/lsgrep/agents/blob/claude/agent-building-lessons-16749f/notebooks/07_memory_and_context.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 7 — Memory and context management

**The claim you should be able to make when you finish:** *"The four strategies
fail differently, not better and worse. I know which one my workload needs,
because I checked whether step 30 needs what step 4 saw."*

Lab 2 proved you have to bound the context: the loop bills quadratically and
only a bound changes that shape. Lab 5 showed the other reason: an unbounded
context degrades the agent that is reading it.

So bounding is not optional. This lab is about what it **costs you**, which
arithmetic cannot tell you — you have to measure the recall.

Twenty-five minutes, no API key.

In [ ]:
# Cell 1 — bootstrap. No GPU, no API key, no spend.
REPO, BRANCH = "https://github.com/lsgrep/agents.git", "claude/agent-building-lessons-16749f"

import os, subprocess, sys

if not os.path.isdir("agents"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "agents", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("agents"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'matplotlib', 'numpy'], check=True)

import agentlab
env = agentlab.notebook_setup()

## 1. The task that makes this hard

A run where facts arrive throughout, and the ones you need at the end are **not**
the recent ones. That is the adversarial-but-realistic case. If the facts you
needed were always recent, "keep the last six results" would win every time and
context engineering would be a solved problem.

In [ ]:
from agentlab.context import make_task

facts = make_task(n_steps=40, n_facts=20, seed=0)
print(f"{'fact':<10} {'arrives at step':>16} {'tokens':>8} {'kind':>10}")
for f in facts[:8]:
    print(f"{f.id:<10} {f.step:>16} {f.tokens:>8,} {'specific' if f.specific else 'thematic':>10}")
print(f"...  {len(facts)} facts, arriving between steps {facts[0].step} and {facts[-1].step}")

The `specific` flag matters more than it looks. A **specific** fact is an
identifier, an exact error string, a number — the kind a summary drops. A
**thematic** fact is "the deploy pipeline is flaky", which survives
summarisation easily. Strategies do not lose information uniformly; they lose
the specifics first, and the specifics are usually what you needed.

## 2. The four strategies

Predict first. Rank these on recall, and separately on token cost:

1. **keep everything** — never drop anything.
2. **clear old results** — keep the last six tool results, drop the rest.
3. **compact** — summarise the transcript when it crosses a threshold.
4. **handles** — put a *reference* in the transcript instead of the payload, and
   re-read when needed.

In [ ]:
from agentlab.context import compare

print(f"{'strategy':<22} {'recall':>8} {'input tokens':>14} {'peak ctx':>10} {'extra calls':>12}")
for o in compare(facts):
    print(f"{o.strategy:<22} {o.recall:>8.0%} {o.input_tokens:>14,} {o.peak_context:>10,} "
          f"{o.extra_calls:>12}")
print()
for o in compare(facts):
    print(f"{o.strategy:<22} {o.note}")

Read the recall column and the token column **together**, because neither alone
picks a winner:

- **keep everything**: perfect recall, biggest bill, and the context that makes
  lab 5's agent fall over. It is not a safe default; it is a different failure.
- **clear**: cheap, trivial to implement, and **30% recall**. It threw away 70%
  of what the run had learned, silently, with no error and no log line.
- **compact**: 70% recall for fewer tokens than clearing. Better — and note
  *which* 30% went.
- **handles**: **100% recall at the lowest token cost of the four.** It is not
  free: those `extra calls` are re-reads, and a re-read is a turn, and a turn
  resends the transcript.

## 3. Why handles change the shape

The other three strategies manage a growing transcript. Handles stop the
transcript from being where the data lives.

Context then grows with the number of things **seen** (one small reference
each) rather than with their **size**, and recall is bounded by whether the agent
knows to look — not by what fits. This is why "just-in-time context" and "the
filesystem is the memory" keep reappearing in production agent designs, and it is
the same instinct as lab 4's advice to bound tool output.

In [ ]:
from agentlab.context import clear_old_results, handles, keep_everything

print(f"{'facts in the run':>17} {'keep all':>12} {'clear':>10} {'handles':>10}")
for n_facts in (10, 20, 40, 80):
    task = make_task(n_steps=60, n_facts=n_facts, seed=1)
    print(f"{n_facts:>17} {keep_everything(task, n_steps=60).peak_context:>12,} "
          f"{clear_old_results(task, n_steps=60).peak_context:>10,} "
          f"{handles(task, n_steps=60).peak_context:>10,}")
print("\nPeak context under 'keep all' grows with the data. Under handles it barely moves.")

## 4. Compaction has a threshold, and thresholds have a failure mode

A compaction trigger set above where your runs actually reach never fires. The
feature ships, the dashboard shows it enabled, and it has never once run.

In [ ]:
from agentlab.context import compact

print(f"{'threshold':>11} {'recall':>8} {'input tokens':>14} {'compactions':>13}")
for threshold in (15_000, 25_000, 40_000, 10_000_000):
    o = compact(facts, threshold=threshold)
    fires = o.extra_calls
    label = f"{threshold:,}" if threshold < 1_000_000 else "way too high"
    print(f"{label:>11} {o.recall:>8.0%} {o.input_tokens:>14,} {fires:>13}")
print("\nThe last row is the bug: recall is perfect and cost is unchanged because")
print("nothing happened. Check your trigger against your runs' actual peak.")

Compaction fidelity is the other parameter, and it is the one to measure rather
than assume. Summarise a real transcript, then count how many identifiers, exact
error strings and numbers survived. It is usually lower than people guess.

In [ ]:
print(f"{'fidelity':>9} {'recall':>8}  what it means")
for fid, meaning in ((0.3, "a terse summary — themes only"),
                     (0.5, "a decent summary"),
                     (0.8, "a summary that was told to preserve identifiers"),
                     (0.95, "structured extraction, not prose")):
    print(f"{fid:>9.0%} {compact(facts, threshold=25_000, fidelity=fid).recall:>8.0%}  {meaning}")
print("\nThe lever is not 'summarise better'. It is 'stop summarising prose and")
print("start extracting fields' — which is a different, more reliable operation.")

## 5. Notes: the cheap durable half

A note is a fact the agent chose to keep, in its own words, at a fraction of the
tokens of the thing it came from. It survives compaction (short, and early), it
survives clearing (you re-inject it), and it survives the session (it is a file).

The failure mode is that the agent has to **decide** to write one, with only the
information it has at the time. Notes are lossy in the same direction as
summaries — they just cost far less.

In [ ]:
from agentlab.context import Notebook

nb = Notebook(max_tokens=300)
for text in ["user's account id is acct_8812 (needed for every write)",
             "the staging DB rejects writes on Sundays — the failing job was cron-4",
             "billing owner is the payments team, not infra",
             "deploy pipeline is flaky in general"]:
    ok = nb.write(text)
    print(f"  [{'kept' if ok else 'FULL'}] {text}")
print(f"\n{nb.tokens} tokens of notes, carried across every compaction the run does.")
print("Compare: the tool results those came from were thousands of tokens.")

## 6. Wiring it into the loop

All of this lives in `on_step` — the hook from lab 1. Here is clearing, in the
harness, as five lines.

In [ ]:
from agentlab.loop import (ModelResponse, PolicyModel, Tool, ToolRegistry, run,
                           text_block, tool_use_block)

KEEP_LAST = 4

def clear_old_tool_results(turn, messages):
    """Replace all but the most recent tool results with a short placeholder."""
    results = [(i, b) for i, m in enumerate(messages) if m["role"] == "user"
               for b in (m["content"] if isinstance(m["content"], list) else [])
               if b.get("type") == "tool_result"]
    for _, block in results[:-KEEP_LAST]:
        if not block.get("_cleared"):
            block["content"] = "[result cleared to save context — call the tool again if needed]"
            block["_cleared"] = True

DOCS = {f"doc-{i}": "lorem ipsum " * 300 for i in range(20)}
tools = ToolRegistry([Tool("read_doc", "Read one document in full.",
                           {"type": "object", "properties": {"doc_id": {"type": "string"}},
                            "required": ["doc_id"]},
                           fn=lambda doc_id: DOCS.get(doc_id, "?"))])

def reader(messages, tools_):
    n = sum(1 for m in messages if m["role"] == "assistant"
            for b in m["content"] if b.get("type") == "tool_use")
    if n < 12:
        return ModelResponse([tool_use_block(f"t{n}", "read_doc", {"doc_id": f"doc-{n}"})], "tool_use")
    return ModelResponse([text_block("done")], "end_turn")

for label, hook in (("no management", None), (f"clear all but last {KEEP_LAST}", clear_old_tool_results)):
    trace = run(PolicyModel(reader), tools, "Read the corpus.", max_steps=15, on_step=hook)
    print(f"{label:<24} peak context {trace.context_high_water:>8,} tokens")

Note the placeholder text. It does not just delete the result — it tells the
model the result *was* there and can be fetched again. A silently vanished tool
result is confusing; an explicitly cleared one is a handle.

## 7. Choosing

There is no default. The question to answer, with your own traces, is:

> **Does step 30 need what step 4 saw?**

- **Rarely** (each step consumes the previous one — a pipeline) → *clear*. It is
  the simplest thing that works, and it is genuinely enough.
- **Sometimes, thematically** (long research, broad synthesis) → *compact*, and
  extract fields rather than summarising prose.
- **Yes, specifically** (debugging, migrations, anything with identifiers) →
  *handles*, plus notes. Do not summarise; the specifics are the job.
- **The run is short** → *keep everything*, and do not pay for machinery lab 2
  showed you would lose money on.

Most real agents want a mix, and the mix is not exotic: handles for the data,
notes for the decisions, clearing for the noise, and compaction as the backstop
when a run outlives all three.

## What you can now say

- *"The four strategies fail differently. Clearing lost 70% of what the run
  learned, silently."*
- *"Compaction keeps the gist and drops the identifiers — so we extract fields
  instead of summarising prose."*
- *"Handles give full recall at the lowest token cost; the price is re-reads,
  and a re-read is a turn."*
- *"A compaction threshold above your runs' peak never fires. Ours is checked
  against the real distribution."*
- *"The question is whether step 30 needs what step 4 saw — and we looked."*

## Next

**[Lab 8](08_fanout.ipynb)** — when the answer is "give it to several agents".